In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the repository scripts directory.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

V3LE_ROOT = Path("/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper")
V3LE_FIG_ROOT = V3LE_ROOT / "figures"
V3LE_DIAG_DIR = V3LE_ROOT / "diag_data"
V3LE_DATA_DIR = V3LE_ROOT / "data"

In [11]:
import os
import glob
import re
import json
import time
import datetime
import xarray as xr
import xcdat as xc
import numpy as np
import pandas as pd
import shutil
import psutil
import subprocess
from pathlib import Path
from typing import Union
# --- safe imports ---
import warnings


In [12]:
def read_exp_dict():    
    run_dict = {
        'NOAA_20C.en00': {'name': 'NOAA_20C.en00', 'period': '185301-201512'},
    }
    return run_dict 

In [13]:
def read_var_dict():
    var_dict = {
            'TS'      : {'unit': 'K',            'min': -6,  'max': 6,  'nlev': 11, },
            'TURFLX'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'SHFLX+LHFLX'}, 
            'LHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, }, 
            'SHFLX'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, }, 
            'FLUT'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNTOA'  : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLUTC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNTOAC' : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FSNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLDS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLNS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            'FLUS'    : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLDS−FLNS'},
            #'FLUSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'form': 'sigma*TS^4'},
            #'FLDSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLNSC+FLUSC'},
            'FLNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'FLNSC'   : {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, 'expr': 'FLDSC−FLUSC'},
            #'SWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'LWCF_SRF': {'unit': 'W m$^{-2}$',   'min': -20, 'max': 20, 'nlev': 11, },
            #'ALBEDOC_SRF' : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13, },
            #'ALBEDO_SRF'  : {'unit': '%',        'min': -6,  'max': 6,  'nlev': 13, },
        }
    return var_dict 

In [14]:
class EnsembleDataProcs:
    def __init__(
        self,
        data_dir,
        file_template,          # e.g., "exp.en%(ENS).monthly_base.YYYYMM-YYYYMM.nc"
        num_ens,
        var_dict,
        out_dir=None,
        chunks=None,
        fontz=12,
        keep_vars=None,
        lat_name="lat",
        lon_name="lon",
        region_defs=None,       # optional custom regions; else uses get_default_region()
    ):
        self.data_dir    = data_dir
        self.out_dir     = out_dir if out_dir else os.path.join(data_dir, "out")
        self.file_tmpl   = file_template
        self.num_ens     = int(num_ens)
        self.var_dict    = var_dict or {}
        self.fontz       = fontz
        self.chunks      = chunks
        self.ds_all      = None
        self.time_coord  = None
        self.keep_vars   = set(keep_vars or ["area"])
        self.lat_name    = lat_name
        self.lon_name    = lon_name
        self.region_defs = region_defs or self.get_default_region()

        # Normalize region longitudes once & keep raw endpoints for full-ring detection
        self._normalize_region_defs_inplace()

        Path(self.out_dir).mkdir(parents=True, exist_ok=True)

    # ---------- Regions ----------
    def get_default_region(self):
        # You can define in either 0..360 or -180..180; we normalize internally
        return {
            "global":   [(-90,  90), (-180, 180)],
            "Atlantic": [(  0,  60), ( 265, 345)],  # will normalize to [-95, -15]
            "Arctic":   [( 50,  90), (-180, 180)],
            "Arctic2":  [( 66.5,90), (-180, 180)],
            "CONUS":    [( 25,  50), ( 235, 295)],
            "Antarctic":[(-90, -50), (-180, 180)],
            "PolarN":   [( 50,  90), (-180, 180)],
            "Greenland":[( 55,  90), ( -75, -10)],
        }

    def _normalize_lon_val(self, x: float) -> float:
        """Normalize a single longitude to [-180, 180)."""
        return ((x + 180.0) % 360.0) - 180.0

    def _normalize_lon_pair(self, lon0: float, lon1: float):
        """
        Normalize a (lon0, lon1) pair conditionally:
          - If both endpoints are in [-180, 180], leave them as-is.
          - If both endpoints are in [0, 360], convert both to [-180, 180).
          - Otherwise (mixed/out-of-range), normalize both (preserves intent for wrap-around).
        """
        in_0_360    = lambda v: 0.0    <= v <= 360.0
        in_m180_180 = lambda v: -180.0 <= v <= 180.0

        both_180 = in_m180_180(lon0) and in_m180_180(lon1)
        both_360 = in_0_360(lon0)    and in_0_360(lon1)

        if both_180:
            return float(lon0), float(lon1)
        if both_360:
            return self._normalize_lon_val(lon0), self._normalize_lon_val(lon1)

        # Mixed/out-of-range: normalize both for consistent semantics
        warnings.warn(f"Mixed/out-of-range longitude pair ({lon0}, {lon1}) "
                      f"— normalizing both to [-180, 180).")
        return self._normalize_lon_val(lon0), self._normalize_lon_val(lon1)

    def _normalize_region_defs_inplace(self):
        """
        Normalize region longitudes only when needed, and stash raw endpoints
        to detect full-ring boxes (e.g., 0..360) later.
        """
        normed = {}
        meta = {}
        for name, ((lat0, lat1), (lon0, lon1)) in self.region_defs.items():
            lon0n, lon1n = self._normalize_lon_pair(float(lon0), float(lon1))
            normed[name] = [(float(lat0), float(lat1)), (lon0n, lon1n)]
            meta[name]   = {"lon_raw": (float(lon0), float(lon1))}
        self.region_defs = normed
        self._region_meta = meta

    def _region_segments_lon(self, lon0n: float, lon1n: float, lon0_raw: float, lon1_raw: float):
        """
        Return lon segments in [-180, 180) covering the requested box.
        Detect full-ring using raw endpoints (difference multiple of 360 and not equal).
        """
        full_ring = (np.isclose(((lon1_raw - lon0_raw) % 360.0), 0.0) and
                     not np.isclose(lon1_raw, lon0_raw))
        if full_ring:
            return [(-180.0, 180.0)]
        if lon0n <= lon1n:
            return [(lon0n, lon1n)]
        else:
            return [(lon0n, 180.0), (-180.0, lon1n)]

    @staticmethod
    def _normalize_dataset_longitude(ds, lon_name="lon"):
        lons = np.asarray(ds[lon_name].values)
        if np.nanmin(lons) >= 0 and np.nanmax(lons) > 180:  # clearly 0..360
            ds = ds.assign_coords({lon_name: ((lons + 180) % 360 - 180)})
            ds = ds.sortby(lon_name)
        return ds

    # ---------- File discovery ----------
    def _ensemble_paths(self):
        """
        Supports templates with '%(ENS)' (e.g., '.en%(ENS).') or without (single file).
        Auto-detects whether ensembles are indexed en00.. or en01..
        """
        paths = []
        has_placeholder = "%(ENS)" in self.file_tmpl
        if has_placeholder:
            # try 1..N first (en01..enNN), else fall back to 0..N-1 (en00..)
            for start, stop in ((0, self.num_ens), (0, self.num_ens)):
                trial = []
                for i in range(start, stop):
                    ens_str = f"{i:02d}"
                    fname = self.file_tmpl.replace("%(ENS)", ens_str)
                    fpath = os.path.join(self.data_dir, fname)
                    if os.path.isfile(fpath):
                        trial.append(fpath)
                    print(self.data_dir, fname)
                if trial:
                    paths = trial
                    break
        else:
            fpath = os.path.join(self.data_dir, self.file_tmpl)
            if os.path.isfile(fpath):
                paths.append(fpath)
            print(self.data_dir, os.path.basename(fpath))

        if not paths:
            raise FileNotFoundError("No ensemble files found after filtering missing files.")
        return sorted(paths)

    # ---------- Basic preprocessing ----------
    def _preprocess(self, ds: xr.Dataset) -> xr.Dataset:
        keep_data = list(ds.data_vars)
        keep_aux  = [v for v in self.keep_vars if v in ds]  # e.g., 'area' or 'areacella'
        out = ds[keep_data + keep_aux]
        for c in ("year", "time", self.lat_name, self.lon_name):
            if c in out and c not in out.coords:
                out = out.set_coords(c)
        # normalize longitudes (do after set_coords)
        if self.lon_name in out.coords:
            out = self._normalize_dataset_longitude(out, self.lon_name)
        return out

    def _detect_time_coord(self, ds: xr.Dataset) -> str:
        if "year" in ds.coords or "year" in ds:
            return "year"
        if "time" in ds.coords or "time" in ds:
            return "time"
        raise ValueError("No 'year' or 'time' coordinate found.")

    @staticmethod
    def _compressible_encoding(ds, complevel=4):
        enc = {}
        for v in ds.data_vars:
            enc[v] = {"zlib": True, "complevel": complevel, "shuffle": True}
            if np.issubdtype(ds[v].dtype, np.floating) and ds[v].dtype == np.float32:
                enc[v]["dtype"] = "float32"
        return enc

    # ---------- Weights ----------
    def _find_area_field(self, ds: xr.Dataset):
        # Common names across E3SM/CMIP/MPAS
        for cand in ("area", "areacella", "areacello", "cell_area", "areaCell", "cellArea"):
            if cand in ds:
                return ds[cand]
        for cand in ("area", "areacella", "areacello"):
            if cand in ds.coords:
                return ds.coords[cand]
        return None

    def _get_weights(self, tmpl: Union[xr.Dataset, xr.DataArray], prefer="area"):
        lat = self.lat_name; lon = self.lon_name
        if isinstance(tmpl, xr.DataArray):
            ds = tmpl.to_dataset(name="_tmp_")
        else:
            ds = tmpl

        if lat not in ds.coords or lon not in ds.coords:
            return None

        if prefer == "area":
            area = self._find_area_field(ds)
            if area is not None:
                w = area
            else:
                prefer = "coslat"

        if prefer == "coslat":
            data_vars = [v for v in ds.data_vars]
            if not data_vars:
                return None
            w = np.cos(np.deg2rad(ds[lat]))
            base = ds[data_vars[0]]
            w, _ = xr.broadcast(w, base.isel({d: 0 for d in base.dims if d not in (lat, lon)}))
            w = w.squeeze(drop=True)

        w = xr.where(np.isfinite(w) & (w > 0), w, 0)
        return w

    def _weights_for_var(self, da: xr.DataArray, weights="auto"):
        if weights in ("auto", "area", "coslat"):
            prefer = "area" if weights in ("auto", "area") else "coslat"
            sample = da
            for d in list(sample.dims):
                if d not in (self.lat_name, self.lon_name):
                    sample = sample.isel({d: 0})
            return self._get_weights(sample, prefer=prefer)
        elif weights is None:
            return None
        else:
            raise ValueError("weights must be 'auto', 'area', 'coslat', or None")

    # ---------- Spatial helpers ----------
    def _coord_range(self, da):
        v = np.asarray(da.values)
        return float(np.nanmin(v)), float(np.nanmax(v))

    def _is_desc(self, da):
        vals = np.asarray(da.values)
        return vals[1] < vals[0] if vals.size > 1 else False

    def _interval_intersection(self, a0, a1, b0, b1):
        lo, hi = max(a0, b0), min(a1, b1)
        return (lo, hi) if lo <= hi else None

    def _subset_region(self, ds: xr.Dataset, regnam: str) -> xr.Dataset:
        """
        Dataset-aware regional subset:
        - uses normalized region defs (−180..180)
        - clips to dataset extents
        - supports descending latitude
        - handles wrap-around by concatenating segments
        """
        if regnam == "global":
            return ds

        (lat0, lat1), (lon0n, lon1n) = self.region_defs[regnam]
        lon0_raw, lon1_raw = self._region_meta[regnam]["lon_raw"]
        lon_segments = self._region_segments_lon(lon0n, lon1n, lon0_raw, lon1_raw)

        if self.lat_name not in ds.coords or self.lon_name not in ds.coords:
            raise ValueError(f"Dataset missing coords '{self.lat_name}'/'{self.lon_name}'")

        # Dataset extents (post-normalization in _preprocess)
        dlat0, dlat1 = self._coord_range(ds[self.lat_name])
        dlon0, dlon1 = self._coord_range(ds[self.lon_name])

        # Latitude intersection, respect coord order
        lat_int = self._interval_intersection(min(lat0, lat1), max(lat0, lat1), dlat0, dlat1)
        if lat_int is None:
            raise ValueError(f"No latitude overlap for region '{regnam}': "
                             f"requested [{lat0},{lat1}], dataset [{dlat0},{dlat1}]")
        la, lb = lat_int
        lat_desc = self._is_desc(ds[self.lat_name])
        lat_slice = slice(lb, la) if lat_desc else slice(la, lb)

        # Longitude intersection per segment
        segs = []
        for a, b in lon_segments:
            inter = self._interval_intersection(a, b, dlon0, dlon1)
            if inter is not None:
                segs.append(inter)

        if not segs:
            raise ValueError(f"No longitude overlap for region '{regnam}': "
                             f"requested {lon_segments}, dataset [{dlon0},{dlon1}]")

        # Warn if clipped
        if (la, lb) != (min(lat0, lat1), max(lat0, lat1)) or len(segs) != len(lon_segments):
            warnings.warn(f"Region '{regnam}' clipped to dataset extent: "
                          f"lat {la:.3f}..{lb:.3f}, lon segments {segs}")

        # Apply selections
        out = ds.sel({self.lat_name: lat_slice})
        parts = [out.sel({self.lon_name: slice(lo, hi)}) for lo, hi in segs]
        return parts[0] if len(parts) == 1 else xr.concat(parts, dim=self.lon_name)

    def _spatial_dims(self, da: xr.DataArray):
        return [d for d in da.dims if d in (self.lat_name, self.lon_name)]

    def _area_mean(self, da: xr.DataArray, weights="auto"):
        spdims = self._spatial_dims(da)
        if not spdims:
            return da  # non-spatial variable
        w = self._weights_for_var(da, weights=weights)
        if w is not None:
            w_sub = w
            if self.lat_name in w_sub.coords and self.lat_name in da.coords and not w_sub[self.lat_name].identical(da[self.lat_name]):
                w_sub = w_sub.sel({self.lat_name: da[self.lat_name]})
            if self.lon_name in w_sub.coords and self.lon_name in da.coords and not w_sub[self.lon_name].identical(da[self.lon_name]):
                w_sub = w_sub.sel({self.lon_name: da[self.lon_name]})
            w_eff = xr.where(np.isfinite(da), w_sub, 0)
            num = (da.where(np.isfinite(da), 0) * w_eff).sum(dim=spdims, skipna=False)
            den = w_eff.sum(dim=spdims)
            out = num / xr.where(den > 0, den, np.nan)
        else:
            out = da.mean(dim=spdims, skipna=True)
        return out
    
    def _is_valid_file(self, path, regions=("global",)):
        try:
            with xr.open_dataset(path, decode_times=False) as ds:
                if "region" in ds.dims and set(regions).issubset(set(ds.region.values)) and len(ds.data_vars) > 0:
                    return True
                else:
                    print(f"[WARN] File exists but missing expected content: {path}")
                    return False
        except Exception as e:
            print(f"[WARN] Unable to open or invalid NetCDF: {path} ({e})")
            return False
        
    # ---------- Read MonthlyBase outputs and write region+global means ----------
    def save_region_and_global_means_from_monthlies(
        self,
        regions=("global",),
        weights="auto",              # 'auto'|'area'|'coslat'|None
        time_slice=None,
        include_nonspatial=True,     # True | "replicate" | False
        compute=True,
        complevel=4,
        output_template="{base}.region_means.nc",  # may include '{base}' and/or '%(ENS)'
    ):
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)
        
        bad_regions = [r for r in regions if r not in self.region_defs]
        if bad_regions:
            raise ValueError(f"Unknown regions: {bad_regions}. Available: {list(self.region_defs.keys())}")

        for in_path in self._ensemble_paths():
            
            # Derive output name
            base = os.path.basename(in_path)
            if base.endswith(".nc"):
                base = base[:-3]

            ens_str = None
            m = re.search(r"\.en(\d{2})\.", base)
            if m:
                ens_str = m.group(1)

            out_fname = output_template
            if "{base}" in out_fname:
                out_fname = out_fname.replace("{base}", base)
            if "%(ENS)" in out_fname and ens_str:
                out_fname = out_fname.replace("%(ENS)", ens_str)

            out_path = os.path.join(self.out_dir, out_fname)
            
            # generate data file 
            if (not os.path.exists(out_path)) or (not self._is_valid_file(out_path, regions)):
                print(f"[INFO] Creating region/global means: {out_path}")
                #ds = xr.open_dataset(in_path, chunks=self.chunks)
                ds = xr.open_dataset(in_path)
                ds = ds.chunk(self.chunks)
                ds = self._preprocess(ds)
                tdim = self._detect_time_coord(ds)
                
                # Optional time slicing
                if time_slice is not None:
                    if isinstance(time_slice, slice):
                        ds = ds.sel({tdim: time_slice})
                    elif isinstance(time_slice, (list, tuple)) and len(time_slice) == 2:
                        ds = ds.sel({tdim: slice(time_slice[0], time_slice[1])})
                    else:
                        raise ValueError("time_slice must be a slice or (start, end)")

                # keep only data vars you’ll process (+ any aux keep_vars that are present)
                proc_vars = [vv for vv in ds.data_vars if vv not in self.keep_vars]
                keep_aux  = [v for v in self.keep_vars if v in ds]
                ds = ds[proc_vars + keep_aux]

                reg_series = {}
                for reg in regions:
                    # 1) Subset region ONCE, then persist to avoid re-slicing per variable
                    ds_reg = ds if reg == "global" else self._subset_region(ds, reg)
                    ds_reg = ds_reg.persist()

                    out_vars = {}

                    # 2) Build weights ONCE per region, using a representative spatial var
                    w_reg = None
                    for _v in proc_vars:
                        spdims = self._spatial_dims(ds_reg[_v])
                        if spdims:
                            # weights='auto' → prefer area, else coslat (your helpers already do this)
                            w_reg = self._weights_for_var(ds_reg[_v], weights=weights)
                            break

                    # 3) Loop variables but reuse ds_reg and w_reg
                    for v in proc_vars:
                        da = ds_reg[v]
                        spdims = self._spatial_dims(da)

                        if spdims:
                            if w_reg is not None:
                                # fast path: xarray’s weighted mean is efficient and numerically stable
                                # make sure weight coords match da (cheap, usually aligns)
                                w_use = w_reg
                                for d in (self.lat_name, self.lon_name):
                                    if (d in da.coords) and (d in w_use.coords) and not w_use[d].identical(da[d]):
                                        w_use = w_use.sel({d: da[d]})
                                area_mean = da.weighted(xr.where(np.isfinite(da), w_use, 0)).mean(dim=spdims)
                            else:
                                area_mean = da.mean(dim=spdims, skipna=True)

                            attrs = dict(da.attrs)
                            ln = attrs.get("long_name", v)
                            attrs["long_name"] = f"Area-averaged {ln} ({reg})"
                            out_vars[v] = area_mean.assign_attrs(attrs)

                        else:
                            if not include_nonspatial:
                                continue
                            if include_nonspatial == "replicate":
                                if reg == regions[0]:   # add once; we’ll broadcast after concat
                                    out_vars[v] = da
                            else:
                                out_vars[v] = da

                    dso = xr.Dataset(out_vars).expand_dims(region=[reg])
                    reg_series[reg] = dso

                # 4) Concatenate once across regions
                out_ds = xr.concat([reg_series[r] for r in regions], dim="region")

                # 5) Replicate non-spatial after concat (if requested)
                if include_nonspatial == "replicate":
                    for v in list(out_ds.data_vars):
                        if "region" not in out_ds[v].dims:
                            out_ds[v] = out_ds[v].expand_dims(region=out_ds.region)

                # 6) Give NetCDF a fighting chance: modest chunks + faster writer
                enc = self._compressible_encoding(out_ds, complevel)
                # suggest chunksizes to reduce tiny blocks (tune as needed)
                for v, e in enc.items():
                    # keep time contiguous; modest lat/lon tiles
                    dims = out_ds[v].dims
                    cs = []
                    for d in dims:
                        if d == "time":
                            cs.append(out_ds.sizes[d])       # single chunk over time
                        elif d == "region":
                            cs.append(out_ds.sizes[d])       # tiny, keep whole
                        elif d in (self.lat_name, self.lon_name):
                            # ~180x180 tiles is a good default; clamp to size
                            target = 180
                            cs.append(min(out_ds.sizes[d], target))
                        else:
                            cs.append(min(out_ds.sizes[d], 512))
                    e["chunksizes"] = tuple(cs)

                # 7) Write (h5netcdf is typically faster & avoids some netCDF4/dask quirks)
                try:
                    out_ds.to_netcdf(out_path, compute=compute, encoding=enc, engine="h5netcdf")
                except Exception:
                    # conservative fallback
                    out_ds = out_ds.unify_chunks().compute()
                    out_ds.to_netcdf(out_path, compute=True, encoding=enc, engine="h5netcdf")

        return None
    

In [15]:
if __name__ == "__main__":
    # --- Paths
    top_dir  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper"
    data_dir = f"{V3LE_DATA_DIR}/global"   # where the monthly_base files actually live
    out_dir  = f"{V3LE_DATA_DIR}/regmn"    # where outputs go
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # --- Vars & ensemble
    var_dict = read_var_dict()
    group     = "NOAA_20C"
    nens      = 1                          # en01..en25
    period    = "1853-2015"
    start_year, end_year = period.split("-")
    date_span = f"{start_year}01-{end_year}12"

    file_template = f"{group}.en%(ENS).monthly_base.{date_span}.nc"

    # One shot: compute both global and Arctic into a 'region' dimension
    regions = ("global", "Arctic","Arctic2")
    time_slice = None
    weights = "coslat"
    chunks = {"time": -1} #{"time": 12, "lat": 180, "lon": 360}
    keep_vars = ["area"] 
    lat_name = "lat" 
    lon_name = "lon" 
    
    proc = EnsembleDataProcs(
        data_dir=data_dir,               # IMPORTANT: this must be where files actually are
        file_template=file_template,
        num_ens=nens,
        var_dict=var_dict,
        out_dir=out_dir,                 # single output location
        chunks=chunks,
        keep_vars=keep_vars,
        lat_name=lat_name,
        lon_name=lon_name,
    )

    # If your class currently appends `regnam` to paths, neutralize that:
    # (do this only if needed)
    proc.data_dir = data_dir
    proc.out_dir  = out_dir
    output_template = f"{group}.en%(ENS).region_means.{date_span}.nc" 
    include_nonspatial = True
    compute = True 
    complevel = 4 
    
    proc.save_region_and_global_means_from_monthlies(
        regions=regions,
        weights=weights,
        time_slice=time_slice,
        include_nonspatial=include_nonspatial,
        compute=compute,
        complevel=complevel,
        output_template=output_template,
    )

    print("[DONE]")
    

/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/global NOAA_20C.en00.monthly_base.185301-201512.nc
[INFO] Creating region/global means: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/data/regmn/NOAA_20C.en00.region_means.185301-201512.nc


/tmp/ipykernel_2453875/3944467610.py:283: UserWarning: Region 'Arctic' clipped to dataset extent: lat 50.000..89.500, lon segments [(-179.5, 179.5)]
  warnings.warn(f"Region '{regnam}' clipped to dataset extent: "
/tmp/ipykernel_2453875/3944467610.py:283: UserWarning: Region 'Arctic2' clipped to dataset extent: lat 66.500..89.500, lon segments [(-179.5, 179.5)]
  warnings.warn(f"Region '{regnam}' clipped to dataset extent: "


[DONE]
